In [1]:
import os
import sys
import json
import gc
from pathlib import Path

# Resolve project root robustly (works even if notebook starts in testings/).
if "PROJECT_ROOT" in globals():
    PROJECT_ROOT = Path(PROJECT_ROOT).resolve()
else:
    _cwd = Path.cwd().resolve()
    _starts = [_cwd] + list(_cwd.parents[:2])
    PROJECT_ROOT = None
    for _start in _starts:
        for _p in [_start] + list(_start.parents):
            if (_p / "src").is_dir():
                PROJECT_ROOT = _p
                break
        if PROJECT_ROOT is not None:
            break
    if PROJECT_ROOT is None:
        raise FileNotFoundError(
            f"Could not locate project root containing 'src' from start={_cwd}"
        )

# Ensure project root is importable and active for relative paths in params.py
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT:", PROJECT_ROOT)

from src import (
    run_pipeline_batch,
    build_ank_superconsolidated_provenance_report,
    generate_consolidated_report_dashboard,
)
from src.params import TB_PATHS, LANG_CODES
from src import filtering as _filtering
from src import prepro as _prepro
from src import findincorpus as _findincorpus
from src import tb2dic as _tb2dic

RUN_ROOT = PROJECT_ROOT / "testings" / "full_allgames_alllangs_dualrun"
RUN_A_DIR = RUN_ROOT / "run_skip_step4"
RUN_B_DIR = RUN_ROOT / "run_with_step4"
FINAL_EXPORT_ROOT = RUN_ROOT / "final_exports"
RUN_A_EXPORT_DIR = FINAL_EXPORT_ROOT / "run_skip_step4"
RUN_B_EXPORT_DIR = FINAL_EXPORT_ROOT / "run_with_step4"


PROJECT_ROOT: C:\Users\Nelso\Documents\MundoDoce\TB2dic


In [ ]:
# Define the parameters for the pipeline
games = ["DOFUS"]  # Specify the game(s) you want to process
#languages = "pt"  # Use "all" to process all languages
output_folder = RUN_B_DIR  # Use the RUN_B_DIR defined in the notebook
final_output_folder = RUN_B_EXPORT_DIR  # Use the RUN_B_EXPORT_DIR defined in the notebook
skip_step_corpusforms = False  # Ensure Step 4 is included
workers = 4  # Set the number of workers to 2
provenance_level = "detailed"
provenance_formats = ["csv", "jsonl"]

# Run the pipeline
pipeline_result = run_pipeline_batch(
    games=games,
    languages=languages,
    sample=0,
    workers=workers,
    batch_size=50,
    add_verb_flags=False,
    quorum=0.5,
    cleanup_stale=True,
    strict_mode=False,
    pair_workers=1,
    skip_step_corpusforms=skip_step_corpusforms,
    output_folder=str(output_folder),
    final_output_folder=str(final_output_folder),
    prewarm_i18n=True,
    provenance_level=provenance_level,
    provenance_formats=provenance_formats,
)

# Print the results
print("Pipeline Result:", json.dumps(pipeline_result, indent=2))

Ignore

In [ ]:

# Set to True to execute only Run B while preserving run_a_out structure in outputs.
RUN_ONLY_B = True

for p in (RUN_ROOT, RUN_A_DIR, RUN_B_DIR, FINAL_EXPORT_ROOT, RUN_A_EXPORT_DIR, RUN_B_EXPORT_DIR):
    p.mkdir(parents=True, exist_ok=True)

all_games = sorted(TB_PATHS.keys())
all_languages = sorted({str(v).strip().lower() for v in LANG_CODES.values() if str(v).strip()})

print("=" * 90)
print("FULL RUN: ALL GAMES + ALL LANGUAGES + DASHBOARDS + ISOLATED FINAL EXPORTS")
print("=" * 90)
print("CWD:", Path.cwd())
print("Games:", all_games)
print("Languages:", all_languages)
print("Run A work dir:", RUN_A_DIR)
print("Run B work dir:", RUN_B_DIR)
print("Run A final export:", RUN_A_EXPORT_DIR)
print("Run B final export:", RUN_B_EXPORT_DIR)
print("Run only B mode:", RUN_ONLY_B)

def _discover_reports(work_dir: Path):
    """Find one preferred report file per (game, lang): prefer CSV over JSONL."""
    reports = {}
    patterns = ["*_token_provenance_report.csv", "*_token_provenance_report.jsonl"]
    for pattern in patterns:
        for path in sorted(work_dir.glob(pattern)):
            stem = path.name
            if not stem.endswith("_token_provenance_report.csv") and not stem.endswith("_token_provenance_report.jsonl"):
                continue
            base = stem.replace("_token_provenance_report.csv", "").replace("_token_provenance_report.jsonl", "")
            if "_" not in base:
                continue
            game, lang = base.rsplit("_", 1)
            key = (game, lang)
            if key not in reports or path.suffix.lower() == ".csv":
                reports[key] = path
    return reports

def _build_dashboards_for_reports(work_dir: Path, title_prefix: str):
    reports = _discover_reports(work_dir)
    dashboard_runs = []
    for (game, lang), report_path in sorted(reports.items()):
        out_html = work_dir / f"{game}_{lang}_token_provenance_report_dashboard.html"
        res = generate_consolidated_report_dashboard(
            report_path=str(report_path),
            output_html_path=str(out_html),
            dashboard_title=f"{title_prefix} - {game}/{lang}",
        )
        dashboard_runs.append({
            "game": game,
            "language": lang,
            "report": str(report_path),
            "dashboard": str(res.get("output_html_path", out_html)),
            "rows": int(res.get("rows", 0)),
            "status": str(res.get("status", "")),
        })
    return dashboard_runs

def _clear_runtime_memory_between_runs():
    """Best-effort release of large in-memory caches before second pass."""
    print("\n" + "~" * 90)
    print("MEMORY CLEANUP BETWEEN RUNS")
    print("~" * 90)

    cache_stats = {}

    try:
        cache_stats["word_form_cache_before"] = len(_filtering._word_form_cache)
        _filtering._word_form_cache.clear()
        _filtering._word_form_cache_hits = 0
        _filtering._word_form_cache_misses = 0
        cache_stats["word_form_cache_after"] = len(_filtering._word_form_cache)
    except Exception as e:
        cache_stats["word_form_cache_error"] = f"{type(e).__name__}: {e}"

    try:
        cache_stats["std_dic_forms_cache_before"] = len(_findincorpus._std_dic_forms_cache)
        _findincorpus._std_dic_forms_cache.clear()
        cache_stats["std_dic_forms_cache_after"] = len(_findincorpus._std_dic_forms_cache)
    except Exception as e:
        cache_stats["std_dic_forms_cache_error"] = f"{type(e).__name__}: {e}"

    try:
        cache_stats["es_i18n_case_cache_before"] = len(_prepro._ES_I18N_CASE_EVIDENCE_CACHE)
        _prepro._ES_I18N_CASE_EVIDENCE_CACHE.clear()
        cache_stats["es_i18n_case_cache_after"] = len(_prepro._ES_I18N_CASE_EVIDENCE_CACHE)
    except Exception as e:
        cache_stats["es_i18n_case_cache_error"] = f"{type(e).__name__}: {e}"

    try:
        cache_stats["prewarm_done_before"] = len(_tb2dic._PIPELINE_PREWARM_DONE)
        _tb2dic._PIPELINE_PREWARM_DONE.clear()
        cache_stats["prewarm_done_after"] = len(_tb2dic._PIPELINE_PREWARM_DONE)
    except Exception as e:
        cache_stats["prewarm_done_error"] = f"{type(e).__name__}: {e}"

    try:
        _findincorpus._LINGUA_DETECTOR_SINGLETON = None
        _findincorpus._LINGUA_DETECTOR_LANGS_SIG = tuple()
        cache_stats["lingua_detector_reset"] = True
    except Exception as e:
        cache_stats["lingua_detector_reset"] = f"error: {type(e).__name__}: {e}"

    collected = gc.collect()
    cache_stats["gc_collected_objects"] = int(collected)

    try:
        import psutil
        proc = psutil.Process(os.getpid())
        rss_mb = proc.memory_info().rss / (1024 ** 2)
        cache_stats["process_rss_mb_after_cleanup"] = round(rss_mb, 2)
    except Exception:
        pass

    print(json.dumps(cache_stats, ensure_ascii=False, indent=2))
    return cache_stats

def _run_pass(pass_name: str, out_dir: Path, final_export_dir: Path, skip_step4: bool, specific_pairs=None):
    print("\n" + "-" * 90)
    print(f"{pass_name}: skip_step_corpusforms={skip_step4}")
    print("-" * 90)

    if specific_pairs:
        games, languages = zip(*specific_pairs)
    else:
        games, languages = all_games, all_languages

    pipeline_result = run_pipeline_batch(
        languages=languages,
        games=games,
        sample=0,
        workers=8,
        batch_size=50,
        add_verb_flags=False,
        quorum=0.5,
        cleanup_stale=True,
        strict_mode=False,
        pair_workers=1,
        skip_step_corpusforms=skip_step4,
        output_folder=str(out_dir),
        final_output_folder=str(final_export_dir),
        prewarm_i18n=True,
        provenance_level="detailed",
        provenance_formats=["csv", "jsonl"],
    )

    discovered = _discover_reports(out_dir)
    if discovered:
        ank_result = build_ank_superconsolidated_provenance_report(
            work_dir=str(out_dir),
            games="all",
            languages="all",
            output_formats=["csv", "jsonl"],
            include_ank_sources=False,
            strict_mode=False,
        )
    else:
        ank_result = {
            "runs": [],
            "summary": {
                "status": "skipped",
                "reason": "no consolidated reports found for ANK merge",
                "work_dir": str(out_dir),
            },
        }

    dashboards = _build_dashboards_for_reports(out_dir, pass_name)

    return {
        "mode": "skip_step4" if skip_step4 else "with_step4",
        "output_dir": str(out_dir),
        "final_export_dir": str(final_export_dir),
        "pipeline_summary": pipeline_result.get("summary", {}),
        "ank_summary": ank_result.get("summary", {}),
        "dashboards_count": len(dashboards),
        "dashboards": dashboards,
    }

if RUN_ONLY_B:
    run_a_out = {
        "mode": "skip_step4",
        "output_dir": str(RUN_A_DIR),
        "final_export_dir": str(RUN_A_EXPORT_DIR),
        "pipeline_summary": {
            "status": "skipped",
            "reason": "Run A execution disabled (RUN_ONLY_B=True)",
        },
        "ank_summary": {"status": "skipped"},
        "dashboards_count": 0,
        "dashboards": [],
    }
else:
    try:
        run_a_out = _run_pass(
            "Pipeline Run A (Skip Step 4)",
            RUN_A_DIR,
            RUN_A_EXPORT_DIR,
            True,
        )
    except Exception as e:
        run_a_out = {
            "mode": "skip_step4",
            "output_dir": str(RUN_A_DIR),
            "final_export_dir": str(RUN_A_EXPORT_DIR),
            "pipeline_summary": {"status": "failed", "error": f"{type(e).__name__}: {e}"},
            "ank_summary": {"status": "skipped"},
            "dashboards_count": 0,
            "dashboards": [],
        }

if RUN_ONLY_B:
    memory_cleanup_stats = {
        "status": "skipped",
        "reason": "Run A was not executed",
    }
else:
    memory_cleanup_stats = _clear_runtime_memory_between_runs()

try:
    completed_pairs = _discover_reports(RUN_B_DIR)
    completed_games_languages = set(completed_pairs.keys())
    all_games_languages = {(game, lang) for game in all_games for lang in all_languages}
    missing_pairs = all_games_languages - completed_games_languages

    run_b_out = _run_pass(
        "Pipeline Run B (With Step 4)",
        RUN_B_DIR,
        RUN_B_EXPORT_DIR,
        False,
        specific_pairs=missing_pairs if missing_pairs else None,
    )
except Exception as e:
    run_b_out = {
        "mode": "with_step4",
        "output_dir": str(RUN_B_DIR),
        "final_export_dir": str(RUN_B_EXPORT_DIR),
        "pipeline_summary": {"status": "failed", "error": f"{type(e).__name__}: {e}"},
        "ank_summary": {"status": "skipped"},
        "dashboards_count": 0,
        "dashboards": [],
    }

final_output = {
    "run_a": run_a_out,
    "memory_cleanup": memory_cleanup_stats,
    "run_b": run_b_out,
}

print("\n" + "=" * 90)
print("RUN COMPLETE")
print("=" * 90)
print("Run A summary:", final_output["run_a"]["pipeline_summary"])
print("Run A final export:", final_output["run_a"]["final_export_dir"])
print("Memory cleanup stats:", final_output["memory_cleanup"])
print("Run B summary:", final_output["run_b"]["pipeline_summary"])
print("Run B final export:", final_output["run_b"]["final_export_dir"])
print("Run A dashboards:", final_output["run_a"]["dashboards_count"])
print("Run B dashboards:", final_output["run_b"]["dashboards_count"])

summary_json = RUN_ROOT / "dualrun_summary.json"
with open(summary_json, "w", encoding="utf-8") as f:
    json.dump(final_output, f, ensure_ascii=False, indent=2)

print("Summary JSON:", summary_json)

## DOFUS PT - Step 4 low-memory run
Use this cell pair when full Step 4 runs exhaust RAM. It trades some reporting detail for much lower memory usage.

In [2]:
# Low-memory single-pair run for DOFUS/PT with Step 4 enabled.
# Main memory savers:
# - step4_compact_corpus_map=True  -> one true-case variant per token in corpus map
# - step4_std_dic_mode='off'       -> skip standard-dic in-memory set build
# - step4_retain_known_forms=False -> keep only NEW forms from Step 4
# - small workers/batch + periodic cache clear reduce peaks

DOFUS_PT_LOW_MEM_OUT = RUN_ROOT / "dofus_pt_lowmem"
DOFUS_PT_LOW_MEM_EXPORT = FINAL_EXPORT_ROOT / "dofus_pt_lowmem"
DOFUS_PT_LOW_MEM_OUT.mkdir(parents=True, exist_ok=True)
DOFUS_PT_LOW_MEM_EXPORT.mkdir(parents=True, exist_ok=True)

dofus_pt_lowmem_result = run_pipeline_batch(
    games=["DOFUS"],
    languages=["pt"],
    sample=0,
    workers=4,
    batch_size=10,
    add_verb_flags=False,
    quorum=0.5,
    cleanup_stale=True,
    strict_mode=False,
    pair_workers=1,
    skip_step_corpusforms=False,
    output_folder=str(DOFUS_PT_LOW_MEM_OUT),
    final_output_folder=str(DOFUS_PT_LOW_MEM_EXPORT),
    prewarm_i18n=True,
    provenance_level="light",
    provenance_formats=["csv", "jsonl"],
    step4_compact_corpus_map=True,
    step4_std_dic_mode="off",
    step4_retain_known_forms=False,
    step4_wordform_cache_max=5000,
    step4_clear_wordform_cache_every_batches=100,
)

print(json.dumps(dofus_pt_lowmem_result.get("summary", {}), ensure_ascii=False, indent=2))
print("Output folder:", DOFUS_PT_LOW_MEM_OUT)
print("Final export folder:", DOFUS_PT_LOW_MEM_EXPORT)

BATCH PIPELINE (STEPS 1-5)
Games     : ['DOFUS']
Languages : ['pt']
Work dir  : C:\Users\Nelso\Documents\MundoDoce\TB2dic\testings\full_allgames_alllangs_dualrun\dofus_pt_lowmem
Options   : sample=0, workers=4, batch_size=10, add_verb_flags=False, quorum=0.5, cleanup_stale=True, strict_mode=False, pair_workers=1, skip_step_corpusforms=False, provenance_level=light, provenance_formats=['csv', 'jsonl'], step4_compact_corpus_map=True, step4_std_dic_mode=off, step4_retain_known_forms=False, step4_wordform_cache_max=5000, step4_clear_wordform_cache_every_batches=100
Dictionary preflight:
  ✅ pt: dics\pt_dic\pt_BR\pt_BR.dic

--------------------------------------------------------------------------------
▶ Processing pair: game=DOFUS | language=pt
--------------------------------------------------------------------------------
Removed stale outputs: 1
Prewarm skipped for DOFUS/pt [cached]
📂 Loading terminology from: TB_ANK_202507/DOFUS_TB.csv
📊 Shape: (59967, 7) (rows, columns)
📋 Columns: ['

In [2]:
# ANK superconsolidated merge from existing Step 4 reports
CONSOLIDATED_INTERMEDIARY_OUTPUT = RUN_A_DIR
CONSOLIDATED_FINAL_OUTPUT = RUN_A_EXPORT_DIR
CONSOLIDATED_INTERMEDIARY_OUTPUT.mkdir(parents=True, exist_ok=True)
CONSOLIDATED_FINAL_OUTPUT.mkdir(parents=True, exist_ok=True)

report_files = sorted(CONSOLIDATED_INTERMEDIARY_OUTPUT.glob("*_token_provenance_report.*"))
print("Discovered report files:", len(report_files))
if report_files:
    print("Sample report files:")
    for p in report_files[:5]:
        print(" -", p.name)

ank_result = build_ank_superconsolidated_provenance_report(
    work_dir=str(CONSOLIDATED_INTERMEDIARY_OUTPUT),
    games="all",
    languages="all",
    output_formats=["csv", "jsonl"],
    include_ank_sources=False,
    strict_mode=False,
 )

print("ANK Result Summary:")
print(json.dumps(ank_result.get("summary", {}), ensure_ascii=False, indent=2))
print("Output dir:", CONSOLIDATED_INTERMEDIARY_OUTPUT)
print("Final export dir:", CONSOLIDATED_FINAL_OUTPUT)

Discovered report files: 72
Sample report files:
 - ANK_de_token_provenance_report.csv
 - ANK_de_token_provenance_report.jsonl
 - ANK_en_token_provenance_report.csv
 - ANK_en_token_provenance_report.jsonl
 - ANK_es_token_provenance_report.csv
ANK Result Summary:
{
  "status": "ok",
  "work_dir": "C:\\Users\\Nelso\\Documents\\MundoDoce\\TB2dic\\testings\\full_allgames_alllangs_dualrun\\run_skip_step4",
  "selected_games": [
    "ANKANIMATION",
    "DOFUS",
    "ONE_MORE_GATE",
    "RETRO",
    "TOUCH",
    "WAKFU",
    "WAVEN"
  ],
  "selected_languages": [
    "de",
    "en",
    "es",
    "fr",
    "pt"
  ],
  "processed_languages": 5,
  "ok_languages": 5,
  "skipped_languages": 0,
  "failed_languages": 0
}
Output dir: C:\Users\Nelso\Documents\MundoDoce\TB2dic\testings\full_allgames_alllangs_dualrun\run_skip_step4
Final export dir: C:\Users\Nelso\Documents\MundoDoce\TB2dic\testings\full_allgames_alllangs_dualrun\final_exports\run_skip_step4


In [3]:
# Build dashboards for ANK superconsolidated reports
ank_csv_reports = sorted(CONSOLIDATED_INTERMEDIARY_OUTPUT.glob("ANK_*_token_provenance_report.csv"))
ank_jsonl_reports = sorted(CONSOLIDATED_INTERMEDIARY_OUTPUT.glob("ANK_*_token_provenance_report.jsonl"))

# Prefer CSV when available, fallback to JSONL.
report_map = {}
for rp in ank_jsonl_reports:
    report_map[rp.stem.replace("_token_provenance_report", "")] = rp
for rp in ank_csv_reports:
    report_map[rp.stem.replace("_token_provenance_report", "")] = rp

selected_reports = [report_map[k] for k in sorted(report_map.keys())]
print("ANK consolidated reports found:", len(selected_reports))

dashboard_runs = []
for report_path in selected_reports:
    report_stem = report_path.stem.replace("_token_provenance_report", "")
    lang = report_stem.replace("ANK_", "", 1)
    out_html = CONSOLIDATED_FINAL_OUTPUT / f"{report_stem}_dashboard.html"

    res = generate_consolidated_report_dashboard(
        report_path=str(report_path),
        output_html_path=str(out_html),
        dashboard_title=f"ANK Superconsolidated Provenance Dashboard ({lang})",
    )

    dashboard_runs.append({
        "language": lang,
        "report": str(report_path),
        "dashboard": str(res.get("output_html_path", out_html)),
        "rows": int(res.get("rows", 0)),
        "status": str(res.get("status", "")),
    })

print("Dashboards generated:", len(dashboard_runs))
for item in dashboard_runs:
    print(f" - [{item['status']}] {item['language']} -> {item['dashboard']} (rows={item['rows']})")

ANK consolidated reports found: 5
Dashboards generated: 5
 - [ok] de -> C:\Users\Nelso\Documents\MundoDoce\TB2dic\testings\full_allgames_alllangs_dualrun\final_exports\run_skip_step4\ANK_de_dashboard.html (rows=29833)
 - [ok] en -> C:\Users\Nelso\Documents\MundoDoce\TB2dic\testings\full_allgames_alllangs_dualrun\final_exports\run_skip_step4\ANK_en_dashboard.html (rows=24050)
 - [ok] es -> C:\Users\Nelso\Documents\MundoDoce\TB2dic\testings\full_allgames_alllangs_dualrun\final_exports\run_skip_step4\ANK_es_dashboard.html (rows=26646)
 - [ok] fr -> C:\Users\Nelso\Documents\MundoDoce\TB2dic\testings\full_allgames_alllangs_dualrun\final_exports\run_skip_step4\ANK_fr_dashboard.html (rows=25461)
 - [ok] pt -> C:\Users\Nelso\Documents\MundoDoce\TB2dic\testings\full_allgames_alllangs_dualrun\final_exports\run_skip_step4\ANK_pt_dashboard.html (rows=25981)
